In [1]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

# Load your code.env file
env_file = r"C:\Users\Abhijeet\code.env"

with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

print(f" Loaded from: {env_file}")

ok_openai = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI : {' Loaded' if ok_openai else ' Not found'}")

 Loaded from: C:\Users\Abhijeet\code.env
OpenAI :  Loaded


In [3]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"],
               capture_output=True)

# Load from code.env
env_file = r"C:\Users\Abhijeet\code.env"
with open(env_file, "r") as f:
    for line in f:
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip().strip('"').strip("'")

# Anthropic not needed — set dummy so scraper skips quietly
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = "not-used"

ok = len(os.environ.get("OPENAI_API_KEY", "")) > 20
print(f"OpenAI key : {' Ready' if ok else ' Not found — check code.env'}")
if ok:
    print(" Ready to run scraper")

OpenAI key :  Ready
 Ready to run scraper


In [5]:
import asyncio
import re
import sys
import json
import urllib.request
import os
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime


# ── SENTENCE-SAFE TRUNCATION ─────────────────────────────────────────────────

def clean_to_sentences(text: str, max_sentences: int = 2) -> str:
    """
    Return at most `max_sentences` complete sentences from text.
    Never cuts mid-sentence. Strips trailing incomplete fragments.
    """
    if not text:
        return ""
    text = re.sub(r'\s+', ' ', text).strip()
    # Split on sentence-ending punctuation followed by space or end-of-string
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # Keep only complete sentences: must end with punctuation and have ≥6 words
    good = [
        s.strip() for s in sentences
        if len(s.strip().split()) >= 6 and re.search(r'[.!?]$', s.strip())
    ]
    return " ".join(good[:max_sentences])


# ── API KEY VALIDATION ────────────────────────────────────────────────────────
# Loads keys from .env file automatically — safe, never hardcode keys in code

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads .env file from same folder as this script
except ImportError:
    pass  # dotenv not installed — keys must be set via os.environ manually

def _get_openai_key() -> str:
    return os.environ.get("OPENAI_API_KEY", "")

def _get_anthropic_key() -> str:
    return os.environ.get("ANTHROPIC_API_KEY", "")

def _check_key(key: str, name: str) -> bool:
    if not key or key.startswith("YOUR_") or len(key) < 20:
        print(f"  ⚠ {name} not set — add it to your .env file")
        return False
    return True

# ── HELPERS 

async def highlight_and_click(page, selector_or_locator, description="Action"):
    try:
        target = (
            page.locator(selector_or_locator).first
            if isinstance(selector_or_locator, str)
            else selector_or_locator
        )
        if await target.is_visible(timeout=2000):
            await target.evaluate(
                "el => { el.style.border = '4px solid #00FF00';"
                " el.style.backgroundColor = 'rgba(0,255,0,0.2)'; }"
            )
            await target.click()
            return True
    except Exception:
        pass
    return False


async def handle_cookies_automatically(page):
    cookie_selectors = [
        "text=Accept All", "text=Accept",
        "button:has-text('Accept')", "button:has-text('OK')",
        "button:has-text('I Agree')", "button:has-text('Close')",
        "button:has-text('Got it')", "button:has-text('Agree')",
    ]
    for selector in cookie_selectors:
        if await highlight_and_click(page, selector, "Cookie Banner"):
            break


async def smart_goto(page, url, timeout=90000):
    """Try networkidle first; fall back to domcontentloaded + commit."""
    try:
        await page.goto(url, wait_until="networkidle", timeout=timeout)
        return True
    except Exception as e:
        print(f"  ⚠ networkidle failed ({e.__class__.__name__}), retrying…")
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
        await asyncio.sleep(6)
        try:
            await page.wait_for_function(
                "() => document.body && document.body.innerText.trim().length > 200",
                timeout=15000)
        except Exception:
            pass
        return True
    except Exception as e:
        try:
            await page.goto(url, wait_until="commit", timeout=timeout)
            await asyncio.sleep(8)
            return True
        except Exception:
            pass
        print(f"  ✗ Failed: {e}")
        return False


# ── CITY DETECTION ────────────────────────────────────────────────────────────

CITY_HINTS = {
    # South Florida — most specific first
    "coral gables":     "Coral Gables, FL",
    "coconut grove":    "Coconut Grove, FL",
    "miami beach":      "Miami Beach, FL",
    "miamibeach":       "Miami Beach, FL",
    "miami gardens":    "Miami Gardens, FL",
    "palmetto bay":     "Palmetto Bay, FL",
    "pinecrest":        "Pinecrest, FL",
    "key biscayne":     "Key Biscayne, FL",
    "homestead":        "Homestead, FL",
    "aventura":         "Aventura, FL",
    "brickell":         "Miami, FL",
    "wynwood":          "Miami, FL",
    "kendall":          "Miami, FL",
    "hialeah":          "Hialeah, FL",
    "doral":            "Doral, FL",
    "boca raton":       "Boca Raton, FL",
    "fort lauderdale":  "Fort Lauderdale, FL",
    "west palm beach":  "West Palm Beach, FL",
    "palm beach":       "Palm Beach, FL",
    "bradenton":        "Bradenton, FL",
    "sarasota":         "Sarasota, FL",
    "orlando":          "Orlando, FL",
    "tampa":            "Tampa, FL",
    "miami":            "Miami, FL",
    "fl 33":            "Miami, FL",
    "fl 34":            "Miami, FL",
    ".mdc.edu":         "Miami, FL",
    ".fl.gov":          "Miami, FL",
    # General Florida fallback
    "florida":          "Miami, FL",
    # Prevent wrong US city matches
    "new york":         "New York, US",
    "los angeles":      "Los Angeles, US",
    "boston":           "Boston, US",
    # International
    "london":           "London, UK",
    "dubai":            "Dubai, UAE",
}

def detect_city(url: str, body_text: str) -> str:
    url_lower = url.lower()
    # Longest hint wins
    for hint in sorted(CITY_HINTS, key=len, reverse=True):
        if hint in url_lower:
            return CITY_HINTS[hint]
    sample = body_text[:4000].lower()
    for hint in sorted(CITY_HINTS, key=len, reverse=True):
        if hint in sample:
            return CITY_HINTS[hint]
    return "Miami, FL"


# ── GARBAGE SITE DETECTION ────────────────────────────────────────────────────
# Skip sites that have been redirected to domain-sale/parking pages

GARBAGE_SIGNALS = [
    "hugedomains", "domain for sale", "buy this domain", "sedoparking",
    "this domain is for sale", "godaddy", "namecheap parking",
    "page not found", "account suspended", "coming soon",
    "under construction", "website coming soon",
]

def is_garbage_site(name: str, body_text: str) -> bool:
    combined = (name + " " + body_text[:500]).lower()
    return any(sig in combined for sig in GARBAGE_SIGNALS)


# ── JUNK PARAGRAPH FILTER ─────────────────────────────────────────────────────
# Sentences that should never appear as data — error messages, UI strings, legal boilerplate

JUNK_PATTERNS = [
    r"the site you are about to visit",
    r"page you requested no longer exists",
    r"managed independently",
    r"beginning of dialog window",
    r"escape will cancel",
    r"^d after this time",
    r"^d to new applications",
    r"^d when the permitted",
    r"^s, refine my essays",
    r"enquire now$",
    r"apply now$",
    r"watch the video",
    r"learn more$",
    r"find out more$",
    r"read more$",
    r"click here",
    r'"school" means the school',
    r"student.*means the child",
    r"parents.*guardians.*applying",
    r"single.elimination system",
    r"park ventures ecoplex",
    r"57 wireless road",
    r"please fill out the form",
    r"fill the registration form",
    r"enrol your child in \d+ easy steps",
    r"step \d+ - complete",
    r"^why do we do what we do",
    r"see our record.breaking class",
    r"view now we have helped",
    r"not sponsored by.*affiliated",
    r"mailing list.*keep you up to date",
    r"joining our mailing list",
    r"over 1\.25 million",
    r"million visitor",
    r"million patron",
    r"to progress your application we require",
    r"please see below the list of documents",
    r"documents that are mandatory to upload",
    r'"school" means the school providing',
    r"our system is limited to uploading",
    r"scans of multi-page documents",
    r"report lost items",
    r"recover belongings",
    r"tourist visa.*online",
    r"apply for your tourist visa",
]

def is_junk(text: str) -> bool:
    if not text:
        return True
    tl = text.lower().strip()
    return any(re.search(p, tl) for p in JUNK_PATTERNS)


# ── IMAGE HARVESTING (module-level — works on homepage AND all subpages) ──────

SKIP_IMG = [
    "logo", "icon", "svg", "1x1", "pixel", "spinner", "placeholder",
    "blank", "transparent", "avatar", "flag", "arrow", "bullet",
    "check", "star", "rating", "rs/facebook", "rs/linkedin", "rs/youtube",
    "loupe", "panier", "menu", "profil", "derouler", "derouler",
    "default_image", "black-beacon", "weather/128",
]
EXTS_IMG = [".jpg", ".jpeg", ".png", ".webp"]

async def harvest_images(pg, base_url: str, image_set: set):
    """Collect all content images from current page into image_set."""
    try:
        # Scroll slowly to trigger lazy-loaded images
        await pg.evaluate("""async () => {
            await new Promise(resolve => {
                let total = 0;
                const dist = 400;
                const timer = setInterval(() => {
                    window.scrollBy(0, dist);
                    total += dist;
                    if (total >= document.body.scrollHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }
                }, 80);
            });
        }""")
        await asyncio.sleep(1)
    except Exception:
        pass

    try:
        # 1. <img> tags — all lazy-load attrs
        for img in await pg.query_selector_all("img"):
            for attr in ["src", "data-src", "data-lazy-src", "data-original",
                         "data-image", "data-bg", "data-lazy", "data-srcset"]:
                c = await img.get_attribute(attr)
                if c:
                    src = c.strip().split(" ")[0].split(",")[0].strip()
                    full = urljoin(base_url, src)
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)
                            and len(full) > 25):
                        image_set.add(full)
            # srcset — take largest (last) candidate
            srcset = await img.get_attribute("srcset")
            if srcset:
                parts = [p.strip().split(" ")[0] for p in srcset.split(",") if p.strip()]
                if parts:
                    full = urljoin(base_url, parts[-1])
                    if (any(e in full.lower() for e in EXTS_IMG)
                            and not any(s in full.lower() for s in SKIP_IMG)):
                        image_set.add(full)

        # 2. og:image + twitter:image
        for sel in ['meta[property="og:image"]', 'meta[name="twitter:image"]',
                    'meta[property="og:image:url"]']:
            try:
                for og in await pg.evaluate(
                    f"() => Array.from(document.querySelectorAll('{sel}')).map(m=>m.content)"
                ):
                    if og and og.strip() and not any(s in og.lower() for s in SKIP_IMG):
                        image_set.add(og.strip())
            except Exception:
                pass

        # 3. CSS background-image on hero/banner/slider/section
        bg_urls = await pg.evaluate("""() => {
            const hits = [];
            const sels = [
                '[style*="background-image"]','[class*="hero"]','[class*="banner"]',
                '[class*="slider"]','[class*="slide"]','[class*="bg-image"]',
                '[class*="cover"]','section','header'
            ];
            sels.forEach(sel => {
                try {
                    document.querySelectorAll(sel).forEach(el => {
                        const s = el.style.backgroundImage ||
                                  getComputedStyle(el).backgroundImage || '';
                        const m = s.match(/url\\(["']?([^"')]+)["']?\\)/);
                        if (m && m[1]) hits.push(m[1]);
                    });
                } catch(e) {}
            });
            return [...new Set(hits)];
        }""")
        for bg in bg_urls:
            if bg:
                full = urljoin(base_url, bg.strip())
                if (any(e in full.lower() for e in EXTS_IMG)
                        and not any(s in full.lower() for s in SKIP_IMG)):
                    image_set.add(full)
    except Exception:
        pass  # Never crash on image errors


def generate_fallback(metric, text_pool):
    metric_keywords = {
        "Coaching Credentials":   ["qualified teacher", "certified", "credentials", "faculty", "expertise"],
        "Student Wellbeing":      ["wellbeing", "pastoral", "counselor", "mental health", "welfare"],
        "Academic Integration":   ["curriculum", "academic", "igcse", "a-level", "ib", "programme"],
        "Competitive Pathway":    ["university", "destination", "results", "examination", "achievement"],
        "Facilities & Resources": ["facilities", "campus", "library", "laboratory", "pool", "gymnasium"],
        "Ongoing Accountability": ["progress", "tracking", "monitoring", "feedback", "report"],
    }
    kws     = metric_keywords.get(metric, [])
    pool    = [p for p in text_pool if not is_junk(p)]
    matched = [p for p in pool if any(k in p.lower() for k in kws)]
    raw     = " ".join(matched[:2]) if matched else (" ".join(pool[:2]) if pool else "")
    return clean_to_sentences(raw, 2)


# ── QUOTE SCRAPER

def _word_count(text):
    return len(text.split())


def scrape_quote_from_soup(soup):
    SKIP_PHRASES = [
        "cookie", "menu", "nav", "©", "privacy", "terms",
        "subscribe", "sign up", "click", "read more", "javascript",
        "khda", "rated", "inspection", "watch the video", "watch video",
        "listen to", "testimonial", "share", "follow us", "contact us",
        "apply now", "find out more", "learn more", "get in touch",
        "download", "register", "book a", "visit us",
        # Legal / UI junk
        "means the child", "parents/guardians", "dialog window",
        "escape will cancel", "site you are about to visit",
        "no longer exists", "managed independently",
    ]
    selectors = [
        "blockquote", "q",
        "[class*='quote']", "[class*='pullquote']",
        "[class*='callout']",
        "[class*='hero'] p", "[class*='banner'] p",
        "[class*='mission'] p", "[class*='vision'] p",
        "figcaption",
    ]
    candidates = []
    for sel in selectors:
        for el in soup.select(sel):
            text = re.sub(r'\s+', ' ', el.get_text()).strip()
            text = text.strip('\u201c\u201d\u2018\u2019"\'')
            wc   = _word_count(text)
            if wc < 8 or wc > 35:
                continue
            if any(x in text.lower() for x in SKIP_PHRASES):
                continue
            if is_junk(text):
                continue
            # Must end with real sentence punctuation
            if not re.search(r'[.!?]$', text):
                continue
            candidates.append(text)
    if not candidates:
        return ""
    candidates.sort(key=lambda t: _word_count(t))
    return candidates[0]


# ── AI QUOTE — uses OpenAI (no Anthropic key needed)

def generate_ai_quote(school_name, school_type, city, about_text):
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return f"Excellence, integrity, and curiosity — the pillars of {school_name}."
    prompt = (
        f"Write ONE short inspirational quote (1 sentence, 10–20 words) "
        f"that reflects the philosophy of {school_name}, a {school_type} in {city}. "
        f"Context: {about_text[:300]}. "
        f"Sound like the organisation's mission statement. "
        f"Return ONLY the quote text — no quotation marks, no attribution, no explanation."
    )
    payload = json.dumps({
        "model":       "gpt-4o",
        "max_tokens":  60,
        "temperature": 0.7,
        "messages":    [{"role": "user", "content": prompt}],
    }).encode("utf-8")
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=payload,
        headers={
            "Content-Type":  "application/json",
            "Authorization": f"Bearer {key}",
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data  = json.loads(resp.read())
            quote = data["choices"][0]["message"]["content"].strip().strip('"\'')
            words = quote.split()
            if len(words) > 25:
                quote = " ".join(words[:25])
            print(f"  ✓ AI quote generated for {school_name}")
            return quote
    except Exception as e:
        print(f"  ⚠ AI quote failed: {e}")
    return f"Excellence, integrity, and curiosity — the pillars of {school_name}."


# ── OpenAI gap-fill ───────────────────────────────────────────────────────────

def openai_fill_missing_fields(results: dict) -> dict:
    """
    After scraping, call OpenAI gpt-4o to fill any field that is still
    empty, 'N/A', or a placeholder.  Only missing fields are sent;
    already-scraped values are NEVER overwritten.
    """
    key = _get_openai_key()
    if not _check_key(key, "OPENAI_API_KEY"):
        return results

    entity_type = results.get("Type", "Independent School")
    is_school   = any(w in entity_type.lower() for w in ["school", "academy", "college", "education"])

    FILL_FIELDS = {
        "Founded":    "Year this organisation was founded (just the 4-digit year). If unknown reply N/A.",
        "Ages":       ("Student age range (e.g. '3–18'). If unknown reply N/A." if is_school
                       else "Age range of members/clients served. If not applicable reply N/A."),
        "Students":   ("Approximate enrolled students (e.g. '1200'). If unknown reply N/A." if is_school
                       else "Approximate number of members/clients/visitors annually. If unknown reply N/A."),
        "Ratio":      "Staff-to-student or staff-to-client ratio. If unknown reply N/A.",
        "Fees":       "Annual fee or membership cost with currency. If unknown reply 'See website'.",
        "About":      f"2 sentences max (≤40 words) describing {results.get('Name','this organisation')}'s identity and mission.",
        "Philosophy": f"2 sentences max (≤40 words) on {results.get('Name','this organisation')}'s approach or ethos.",
        "Outcomes":   f"2 sentences max (≤40 words) on what {results.get('Name','this organisation')} achieves for its members.",
        "Admissions": f"2 sentences max (≤40 words) on how to join or enrol at {results.get('Name','this organisation')}.",
        "Tagline":    f"One compelling tagline (≤20 words) for {results.get('Name','this organisation')}.",
    }

    PERF_FIELDS = {
        "Coaching Credentials":   f"2 UNIQUE sentences (≤35 words) on staff qualifications or expertise at {results.get('Name','')}.",
        "Student Wellbeing":      f"2 UNIQUE sentences (≤35 words) on how {results.get('Name','')} supports member/student wellbeing.",
        "Academic Integration":   f"2 UNIQUE sentences (≤35 words) on the programmes or curriculum at {results.get('Name','')}.",
        "Competitive Pathway":    f"2 UNIQUE sentences (≤35 words) on achievements, results or pathways at {results.get('Name','')}.",
        "Facilities & Resources": f"2 UNIQUE sentences (≤35 words) on facilities, venues or resources at {results.get('Name','')}.",
        "Ongoing Accountability": f"2 UNIQUE sentences (≤35 words) on how {results.get('Name','')} tracks progress or quality.",
    }

    # Build context from whatever was scraped successfully
    ctx_lines = []
    for k in ["Name", "Type", "City", "URL", "Tagline", "About", "Philosophy",
               "Outcomes", "Admissions", "Founded", "Ages", "Students", "Ratio", "Fees"]:
        v = results.get(k, "")
        if v:
            ctx_lines.append(f"{k}: {v}")
    for pk, pv in results.get("Performance", {}).items():
        if pv:
            ctx_lines.append(f"{pk}: {pv[:200]}")
    context = "\n".join(ctx_lines) or f"URL: {results.get('URL', '')}"

    # Decide which fields still need filling
    missing_main = {
        f: p for f, p in FILL_FIELDS.items()
        if not results.get(f) or results[f] in ("N/A", "See website", "Rolling admissions", "Year-round", "")
    }

    def _perf_needs_fill(val: str, name: str) -> bool:
        if not val or val.strip() in ("N/A", "", "N/a"):
            return True
        val_clean = re.sub(r'\s+', ' ', val).strip()
        if len(val_clean.split()) < 8:
            return True
        irrelevant = [
            "cookie","privacy policy","gdpr","javascript",
            "seven emirates","fishing village","tourist visa",
            "no longer exists","site you are about to visit",
            # Miami-specific junk
            "document.getelementbyid","ak_js","delta document",
            "accessibility statement","skip to main content",
            "city commission","city manager","commissioner",
            "img academy will provide sheets","pillow and blanket",
            "camper did not receive sheets",
            "your suggestion box",
            "community satisfaction survey",
        ]
        vl = val_clean.lower()
        return any(phrase in vl for phrase in irrelevant)

    missing_perf = {
        f: p for f, p in PERF_FIELDS.items()
        if _perf_needs_fill(results.get("Performance", {}).get(f, ""), f)
    }

    all_missing = {**missing_main, **missing_perf}
    if not all_missing:
        print("  ✓ All fields complete — OpenAI fill not needed.")
        return results
    print(f"  → OpenAI filling {len(all_missing)} missing field(s): {', '.join(all_missing.keys())}")

    # Build one compact prompt
    field_instructions = "\n".join(f'  "{k}": {v}' for k, v in all_missing.items())
    prompt = f"""You are a Miami and South Florida education data analyst with knowledge of FL private schools, academies, and enrichment organisations.
Use the known data below AND your own knowledge to fill the missing fields accurately.

KNOWN DATA:
{context}

Fill ONLY these fields (reply with a JSON object, keys exactly as shown):
{field_instructions}

Rules:
- Use your knowledge of this Miami/Florida institution to fill factual fields (Founded, Ages, Students, Fees in USD).
- Fees: use $ and per year. Miami private school range typically $25,000–$55,000 per year.
- Keep paragraph fields SHORT: max 2 sentences, ≤40 words total per field.
- Each performance field MUST describe a DIFFERENT aspect — no repeated sentences across fields.
- Each field must sound distinct — different vocabulary, different facts, different angle.
- Do NOT add commentary, markdown fences, or extra keys.
- Reply with valid JSON only."""

    payload = json.dumps({
        "model": "gpt-4o",
        "max_tokens": 2000,
        "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}],
    }).encode("utf-8")

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=payload,
        headers={
            "Content-Type":  "application/json",
            "Authorization": f"Bearer {key}",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data  = json.loads(resp.read())
            raw   = data["choices"][0]["message"]["content"].strip()
            # Strip markdown fences if present
            raw   = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.M)
            raw   = re.sub(r"\s*```$",           "", raw, flags=re.M).strip()
            filled = json.loads(raw)

            filled_keys = []
            for fkey, val in filled.items():
                if not val or val in ("N/A", "See website"):
                    continue
                if fkey in FILL_FIELDS:
                    if not results.get(fkey) or results[fkey] in (
                            "N/A", "See website", "Rolling admissions", "Year-round", ""):
                        results[fkey] = str(val).strip()
                        filled_keys.append(fkey)
                elif fkey in PERF_FIELDS:
                    # Overwrite if missing OR if it was junk
                    if _perf_needs_fill(results["Performance"].get(fkey, ""), fkey):
                        results["Performance"][fkey] = str(val).strip()
                        filled_keys.append(fkey)

            if filled_keys:
                print(f"  ✓ OpenAI filled: {', '.join(filled_keys)}")
            else:
                print("  ✓ OpenAI: nothing new to fill.")

    except json.JSONDecodeError as e:
        print(f"  ⚠ OpenAI fill JSON parse error: {e}")
    except Exception as e:
        print(f"  ⚠ OpenAI fill failed: {e}")

    return results


def generate_wp_blocks(r):
    school_name  = r.get("Name", "")
    description  = r.get("Tagline", "")
    age          = r.get("Ages", "")
    students     = r.get("Students", "")
    ratio        = r.get("Ratio", "")
    founded      = r.get("Founded", "")
    school_type  = r.get("Type", "")
    city         = r.get("City", "")
    annual_fee   = r.get("Fees", "")
    about        = r.get("About", "")
    philosophy   = r.get("Philosophy", "")
    outcomes     = r.get("Outcomes", "")
    quote        = r.get("Quote", "")
    website_url  = r.get("URL", "")
    admissions   = r.get("Admissions", "")
    images       = r.get("Images", [])
    perf         = r.get("Performance", {})
    enquiry_open = r.get("EnquiryOpen", "Year-round")
    app_deadline = r.get("AppDeadline", "Rolling admissions")

    website_display = website_url.replace("https://", "").replace("http://", "").rstrip("/")
    city_slug       = city.lower().replace(", ", "-").replace(" ", "-").replace(",", "")
    city_short      = city.split(",")[0].strip()

    img0 = images[0] if len(images) > 0 else ""
    img1 = images[1] if len(images) > 1 else img0
    img2 = images[2] if len(images) > 2 else img0
    img3 = images[3] if len(images) > 3 else img0
    img4 = images[4] if len(images) > 4 else img0

    coaching       = perf.get("Coaching Credentials", "")
    wellbeing      = perf.get("Student Wellbeing", "")
    academic       = perf.get("Academic Integration", "")
    competitive    = perf.get("Competitive Pathway", "")
    facilities     = perf.get("Facilities & Resources", "")
    accountability = perf.get("Ongoing Accountability", "")

    wp = f"""<!-- wp:image {{"id":440,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="" class="wp-image-440"/></figure>
<!-- /wp:image -->

<!-- wp:paragraph -->
<p><a href="elite-home.html"><br>Elite</a>&#8250; <a href="elite-city-{city_slug}.html">{city_short}</a>&#8250; <a href="elite-program-academic.html">Elite Academic Programs</a></p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Kidrovia Elite \u2014 Listed</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Elite Academic Programs</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":1}} -->
<h1 class="wp-block-heading"><em>{school_name}</em></h1>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{description}</p>
<!-- /wp:paragraph -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph -->
<p>{age}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Ages</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph -->
<p>{students}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Students</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph -->
<p>{ratio}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Ratio</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:paragraph -->
<p>{founded}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Founded</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column {{"width":"66.66%"}} -->
<div class="wp-block-column" style="flex-basis:66.66%"><!-- wp:heading -->
<h2 class="wp-block-heading">About <em>{school_name}</em></h2>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{about}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column {{"width":"33.33%"}} -->
<div class="wp-block-column" style="flex-basis:33.33%"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Institution Details</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Type</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Ages</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Students</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Ratio </h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Founded</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">City</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Annual fee</h5>
<!-- /wp:heading --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{school_type}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{age}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{students}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{ratio}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{founded}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{city}</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">{annual_fee}</h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Contact &amp; Enquiry</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading"><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display} &#8594;</a></h5>
<!-- /wp:heading --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">Quotes</h5>
<!-- /wp:heading -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">{school_name}</h5>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Philosophy</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">How they&nbsp;<em>teach</em></h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{philosophy}<br><br><br></p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Outcomes</h5>
<!-- /wp:heading -->

<!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Where students&nbsp;<em>go</em></h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{outcomes}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:image {{"id":441,"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img1}" alt="" class="wp-image-441"/></figure>
<!-- /wp:image --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column {{"width":"100%"}} -->
<div class="wp-block-column" style="flex-basis:100%"><!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:image {{"id":449,"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img2}" alt="" class="wp-image-449"/></figure>
<!-- /wp:image --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:image {{"id":450,"sizeSlug":"large","linkDestination":"none"}} -->
<figure class="wp-block-image size-large"><img src="{img3}" alt="" class="wp-image-450"/></figure>
<!-- /wp:image --></div>
<!-- /wp:column --></div>
<!-- /wp:columns --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:image {{"id":451,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img4}" alt="" class="wp-image-451"/></figure>
<!-- /wp:image --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:image {{"id":452,"sizeSlug":"full","linkDestination":"none"}} -->
<figure class="wp-block-image size-full"><img src="{img0}" alt="" class="wp-image-452"/></figure>
<!-- /wp:image --></div>
<!-- /wp:column --></div>
<!-- /wp:columns --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:heading {{"level":4}} -->
<h4 class="wp-block-heading">Our Assessment</h4>
<!-- /wp:heading -->

<!-- wp:heading -->
<h2 class="wp-block-heading"><strong>How {school_name} performs</strong></h2>
<!-- /wp:heading -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Coaching Credentials</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{coaching}<br></p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Student Wellbeing</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{wellbeing}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Academic Integration</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{academic}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Competitive Pathway</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{competitive}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Facilities &amp; Resources</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{facilities}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":3}} -->
<h3 class="wp-block-heading">Ongoing Accountability</h3>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{accountability}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:columns -->
<div class="wp-block-columns"><!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading -->
<h2 class="wp-block-heading">Admissions &amp;<br><em>How to Apply</em></h2>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{admissions}</p>
<!-- /wp:paragraph --></div>
<!-- /wp:column -->

<!-- wp:column -->
<div class="wp-block-column"><!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Enquiries Open</h5>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{enquiry_open}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Application Deadline</h5>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{app_deadline}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Annual Fee</h5>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{annual_fee}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Location</h5>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p>{city}</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"level":5}} -->
<h5 class="wp-block-heading">Website</h5>
<!-- /wp:heading -->

<!-- wp:paragraph -->
<p><a href="{website_url}" target="_blank" rel="noreferrer noopener">{website_display}</a></p>
<!-- /wp:paragraph --></div>
<!-- /wp:column --></div>
<!-- /wp:columns -->

<!-- wp:paragraph {{"align":"center"}} -->
<p class="has-text-align-center">&ldquo;{quote}&rdquo;</p>
<!-- /wp:paragraph -->

<!-- wp:heading {{"textAlign":"center","level":5}} -->
<h5 class="wp-block-heading has-text-align-center">{school_name}</h5>
<!-- /wp:heading -->
"""
    return wp


async def extract_school_data(url):
    async with async_playwright() as p:
        import platform
        headless = os.environ.get("HEADLESS","true" if platform.system()=="Linux" else "false").lower()=="true"
        browser  = await p.chromium.launch(headless=headless, slow_mo=0 if headless else 200)
        context  = await browser.new_context(viewport={"width": 1280, "height": 800})
        page     = await context.new_page()

        results = {
            "Name": "", "Tagline": "", "Founded": "",
            "City": "",
            "Ages": "", "Students": "", "Ratio": "", "Fees": "",
            "Type": "Independent School",
            "About": "", "Philosophy": "", "Outcomes": "",
            "Admissions": "", "Quote": "",
            "EnquiryOpen": "Year-round",
            "AppDeadline": "Rolling admissions",
            "Performance": {}, "URL": url, "Images": [],
        }

        global_text_pool = []

        FALLBACKS = {
            "Founded":  "",
            "Ages":     "",
            "Students": "",
            "Ratio":    "N/A",
            "Fees":     "See website",
        }

        # Fee patterns — USD only, validated range $1,000–$100,000
        fee_patterns = [
            r'\$\s*([\d,]+)(?:\.\d{2})?\s*per\s*(?:year|annum|semester|month)',
            r'tuition\s*(?:fee)?s?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
            r'annual\s*(?:tuition|fee)s?\s*(?:of|is|:)?\s*\$\s*([\d,]+)',
            r'fee[s]?\s*(?:of|is|are|:)?\s*\$\s*([\d,]+)',
        ]

        def extract_fee_validated(text: str) -> str:
            """Extract USD fee, reject if < $1,000 (likely a bare $ match, not tuition)."""
            for pat in fee_patterns:
                for m in re.finditer(pat, text, re.I):
                    try:
                        num = int(m.group(1).replace(',', ''))
                        if 1000 <= num <= 100000:
                            return m.group(0).strip()
                    except (IndexError, ValueError):
                        pass
            return ""

        # FIX #3: Ratio — added 1:X and X:1 formats common in UAE
        ratio_patterns = [
            r'(\d+\s*:\s*\d+)\s*(?:student[- ]to[- ](?:faculty|teacher)|teacher[- ]to[- ]student|ratio)',
            r'ratio\s*(?:of\s*)?(\d+\s*:\s*\d+)',
            r'(\d+\s*:\s*\d+)\s*ratio',
            r'1\s*(?:teacher|member of staff|coach)\s+(?:to|per|:)\s+(\d+)\s*(?:students?|pupils?|learners?)',
            r'(\d+)\s*(?:students?|pupils?|learners?)\s+(?:per|to)\s+(?:every\s+)?(?:1\s+)?(?:teacher|tutor|coach)',
        ]

        # Age patterns — explicit "aged X to Y" or "grades X to Y" only
        age_patterns = [
            r'students?\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'children\s+aged\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'ages?\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'from\s+age\s+(\d+)\s*(?:to|[\u2013\-])\s*(\d+)',
            r'(\d+)\s*(?:to|[\u2013\-])\s*(\d+)\s*years?\s*(?:old)',
            # US grades — only full grade ranges, not bare "6-8" which is grade range
            r'grades?\s+(?:k|kindergarten|prek|pre-k)?\s*(?:through|to|[\u2013\-])\s*grade\s*(\d+)',
            r'grades?\s+(\d+)\s*(?:through|to|[\u2013\-])\s*(\d+)',
            r'grade\s+(\d+)\s*(?:through|to|[\u2013\-])\s*(?:grade\s+)?(\d+)',
        ]

        # FIX #5: Student patterns
        student_patterns = [
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:pupils?|students?|boys|girls|young people|young women|learners?|scholars?)',
            r'(?:approximately|around|about|some|over|more than)\s+(\d[\d,]+)\s*(?:pupils?|students?|girls?|learners?)',
            r'school\s+of\s+(?:approximately|around|about)?\s*(\d[\d,]+)',
            r'enroll(?:ment|ment|ing)\s+(?:of\s+)?(?:approximately|about)?\s*(\d[\d,]+)',
            r'community\s+of\s+(?:over\s+)?(\d[\d,]+)\s*(?:students?|learners?)',
            r'(\d[\d,]+)\s*(?:\+)?\s*(?:student|pupil|learner|child)',
        ]

        print(f"Connecting to: {url}...")

        loaded = await smart_goto(page, url)
        if not loaded:
            print(f"  ✗ Could not load {url}, skipping.")
            await browser.close()
            return None

        try:
            await handle_cookies_automatically(page)

            # Name — prefer og:site_name / og:title over <title> which can return "Home"
            og_site  = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:site_name\"]'); return m ? m.content : ''; }")
            og_title = await page.evaluate("() => { const m = document.querySelector('meta[property=\"og:title\"]'); return m ? m.content : ''; }")
            raw_title = await page.title()

            REJECT_NAMES = {
                "home","welcome","homepage","index","main","","welcome to",
                "co","test prep tutoring","about",
            }
            REJECT_STARTS = ("welcome to ", "home -", "home |", "top private school")
            REJECT_LONG_PHRASES = [
                "the polo training foundation is dedicated",
                "athletic training & performance camps at",
                "personalization","advertising effectiveness",
            ]

            KNOWN_NAMES_MIA = {
                "ransomeverglades.org":  "Ransom Everglades School",
                "gulliverschools.org":   "Gulliver Schools",
                "gulliverprep.org":      "Gulliver Prep",
                "carrollton.org":        "Carrollton School of the Sacred Heart",
                "palmertrinity.org":     "Palmer Trinity School",
                "miamicountryday.org":   "Miami Country Day School",
                "imgacademy.com":        "IMG Academy",
                "polotraining.org":      "Polo Training Foundation",
                "crandonparktennis.com": "Crandon Park Tennis Center",
                "princetonreview.com":   "The Princeton Review",
                "baptisteyoga.com":      "Baron Baptiste Yoga",
                "miamimusicproject.org": "Miami Music Project",
                "nwsa.mdc.edu":          "New World School of the Arts",
                "miamibeachfl.gov":      "Miami Beach Tennis Center",
            }
            domain = url.split("/")[2].replace("www.", "")
            known = next((v for k, v in KNOWN_NAMES_MIA.items() if k in domain), None)

            def clean_name(candidate: str) -> str:
                parts = re.split(r'\s*[|—]\s*', candidate)
                parts = [p.strip() for p in parts if p.strip()]
                GENERIC = {"home","welcome","index","main","page","site","portal","homepage","co","about",""}
                n = parts[0] if parts else candidate.strip()
                if n.lower() in GENERIC and len(parts) > 1:
                    n = parts[-1].strip()
                for prefix in ("welcome to ",):
                    if n.lower().startswith(prefix):
                        n = n[len(prefix):].strip()
                return n

            for candidate in [og_site, og_title, raw_title]:
                name = clean_name(candidate)
                if (name
                        and name.lower() not in REJECT_NAMES
                        and not any(name.lower().startswith(p) for p in REJECT_STARTS)
                        and not any(bad in name.lower() for bad in REJECT_LONG_PHRASES)
                        and len(name.split()) <= 8):
                    results["Name"] = name
                    break
            if known and (not results["Name"] or results["Name"].lower() in REJECT_NAMES):
                results["Name"] = known
            if not results["Name"]:
                results["Name"] = known or clean_name(raw_title)
            body_text       = await page.inner_text("body")

            # JS-heavy sites: scroll to trigger rendering
            if len(body_text.strip()) < 300:
                print("  ⚠ Page body too short — scrolling to trigger JS render…")
                try:
                    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
                    await asyncio.sleep(3)
                    await page.evaluate("window.scrollTo(0, 0)")
                    await asyncio.sleep(2)
                    body_text = await page.inner_text("body")
                    print(f"  Body after scroll: {len(body_text)} chars")
                except Exception:
                    try:
                        body_text = await page.inner_text("body")
                    except Exception:
                        pass

            results["City"] = detect_city(url, body_text)

            # Skip garbage/parked/domain-sale sites immediately
            if is_garbage_site(results["Name"], body_text):
                print(f"  ✗ Garbage/parked site detected — skipping {url}")
                await browser.close()
                return None

            def extract(pattern):
                m = re.search(pattern, body_text, re.IGNORECASE)
                return m.group(1).strip() if m else ""

            current_year = datetime.now().year

            # ── BUG FIX B: Founded — penalise pre-1900 years and sentences
            #   that are clearly about a UK/parent school, not this campus
            def extract_founded_from_text(text, existing_candidates=None):
                STRONG_KWS = [
                    "founded in", "established in", "was founded", "founding year",
                    "founded by", "foundress", "charter", "incorporated in",
                    "opened in", "opened its doors", "since", "est.",
                ]
                WEAK_KWS  = ["founded", "established", "foundation", "history", "began"]
                SKIP_WORDS = ["copyright", "©", "reserved", "updated", "revised",
                              "class of", "alumni", "graduate", "batch of"]
                # Sentences that refer to a PARENT / FOREIGN campus — not this school
                FOREIGN_CTX = [
                    "in the uk", "in england", "in derbyshire", "in scotland",
                    "in wales", "in ireland", "parent school", "sister school",
                    "original school", "home school",
                    "repton school in", "repton uk",
                    "in derbyshire, uk", "in london, uk",
                ]
                cands = existing_candidates if existing_candidates is not None else []
                clean = re.sub(r'\s+', ' ', text.lower())
                clean = re.sub(r'(copyright|©).*?\d{4}', '', clean)
                for sent in re.split(r'[.!?\n]', clean):
                    if any(w in sent for w in SKIP_WORDS):
                        continue
                    has_strong = any(k in sent for k in STRONG_KWS)
                    has_weak   = any(k in sent for k in WEAK_KWS)
                    if not (has_strong or has_weak):
                        continue
                    is_foreign = any(ctx in sent for ctx in FOREIGN_CTX)
                    for y in re.findall(r'\b(1[6-9]\d{2}|20[012]\d)\b', sent):
                        year = int(y)
                        if not (1600 <= year <= current_year - 1):
                            continue
                        score = 3 if has_strong else 1
                        # Pre-1900: almost certainly a UK/historic parent school
                        if year < 1900:
                            score -= 4
                        # Sentence explicitly about a foreign/parent school
                        if is_foreign:
                            score -= 3
                        # Too recent — probably not a founding year
                        if year >= current_year - 2:
                            score -= 3
                        cands.append((score, year))
                return cands

            founded_candidates = extract_founded_from_text(body_text)
            if founded_candidates:
                founded_candidates.sort(key=lambda x: (-x[0], x[1]))
                results["Founded"] = str(founded_candidates[0][1])

            # Collect ALL valid age ranges from page, pick the WIDEST (most complete)
            age_candidates = []
            for pat in age_patterns:
                for m in re.finditer(pat, body_text, re.I):
                    try:
                        g = m.groups()
                        if len(g) >= 2 and g[0] and g[1]:
                            lo, hi = int(g[0]), int(g[1])
                            if lo < hi and lo >= 2 and hi <= 25:
                                age_candidates.append((lo, hi))
                        elif len(g) == 1 and g[0]:
                            hi = int(g[0])
                            if 10 <= hi <= 25:
                                age_candidates.append((3, hi))
                    except Exception:
                        pass
            if age_candidates:
                # Pick the widest range (highest hi - lo) — gives full school span
                best = max(age_candidates, key=lambda x: x[1] - x[0])
                results["Ages"] = f"{best[0]}\u2013{best[1]}"

            if not results["Ages"]:
                AGE_MAP = {
                    "prek": (4, 5), "pre-k": (4, 5), "pre-kindergarten": (4, 5),
                    "preschool": (3, 5), "nursery": (3, 4),
                    "kindergarten": (5, 6), "kinder": (5, 6),
                    "grade 1": (6, 7), "1st grade": (6, 7),
                    "grade 2": (7, 8), "grade 3": (8, 9), "grade 4": (9, 10),
                    "grade 5": (10, 11), "grade 6": (11, 12), "6th grade": (11, 12),
                    "grade 7": (12, 13), "grade 8": (13, 14),
                    "grade 9": (14, 15), "9th grade": (14, 15),
                    "grade 10": (15, 16), "grade 11": (16, 17),
                    "grade 12": (17, 18), "12th grade": (17, 18),
                    "elementary school": (5, 11), "lower school": (5, 11),
                    "middle school": (11, 14), "upper school": (14, 18),
                    "high school": (14, 18),
                    "college preparatory": (14, 18), "college prep": (14, 18),
                }
                tl  = body_text.lower()
                fy  = [v for k, v in AGE_MAP.items() if k in tl]
                results["Ages"] = f"{min(x[0] for x in fy)}\u2013{max(x[1] for x in fy)}" if fy else ""

            for pat in student_patterns:
                for m in re.finditer(pat, body_text, re.I):
                    num_str = re.sub(r'[\s,]', '', m.group(1))
                    try:
                        num = int(num_str)
                        if not (50 <= num <= 8000):
                            continue
                        ctx = body_text[max(0, m.start()-8):m.end()+8]
                        if '%' in ctx or '.' in m.group(1):
                            continue
                        results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                        break
                    except ValueError:
                        pass
                if results["Students"]:
                    break

            for pat in ratio_patterns:
                m = re.search(pat, body_text, re.I)
                if m:
                    results["Ratio"] = m.group(1).replace(" ", "")
                    break

            results["Fees"] = extract_fee_validated(body_text)

            # Type detection — Miami/Florida specific, URL hardcodes first
            tl = body_text.lower()
            url_l = url.lower()
            if "imgacademy.com" in url_l:
                results["Type"] = "Sports Performance Academy"
            elif "polotraining.org" in url_l or ("polo" in tl and "foundation" in tl):
                results["Type"] = "Polo Training Foundation"
            elif "princetonreview.com" in url_l or "kaplan.com" in url_l:
                results["Type"] = "Academic Test Prep & Tutoring"
            elif "miamimusicproject.org" in url_l:
                results["Type"] = "Youth Music Programme"
            elif "nwsa.mdc.edu" in url_l or "new world school" in tl:
                results["Type"] = "Performing & Visual Arts School"
            elif "baptist" in url_l and "yoga" in url_l:
                results["Type"] = "Yoga & Wellness Studio"
            elif "crandon" in url_l and "tennis" in url_l:
                results["Type"] = "Public Tennis Centre"
            elif "miamibeach" in url_l and "tennis" in url_l:
                results["Type"] = "Public Tennis Centre"
            # General detection
            elif "all-girls" in tl or "girls' school" in tl or "girls school" in tl:
                results["Type"] = "Independent All-Girls School"
            elif "all-boys" in tl or "boys' school" in tl or "boys school" in tl:
                results["Type"] = "Independent Boys' School"
            elif "montessori" in tl:
                results["Type"] = "Montessori School"
            elif "ib world" in tl or "international baccalaureate" in tl:
                results["Type"] = "IB World Day School"
            elif "boarding" in tl and "day" in tl:
                results["Type"] = "Independent Boarding & Day School"
            elif "boarding" in tl:
                results["Type"] = "Independent Boarding School"
            elif "day school" in tl or "day students" in tl:
                results["Type"] = "Independent Day School"
            elif "college prep" in tl or "college preparatory" in tl:
                results["Type"] = "College Preparatory School"
            elif "charter school" in tl:
                results["Type"] = "Charter School"
            elif "magnet school" in tl:
                results["Type"] = "Magnet School"
            elif "tennis" in tl and ("academy" in tl or "center" in tl or "centre" in tl):
                results["Type"] = "Tennis Academy"
            elif "polo" in tl and ("club" in tl or "training" in tl):
                results["Type"] = "Polo Club & Training"
            elif "yoga" in tl and ("studio" in tl or "center" in tl):
                results["Type"] = "Yoga & Wellness Studio"
            elif "music" in tl and ("programme" in tl or "program" in tl or "project" in tl):
                results["Type"] = "Music Programme"
            elif "performing arts" in tl or "visual arts" in tl:
                results["Type"] = "Performing & Visual Arts School"
            elif "sports" in tl and ("academy" in tl or "camp" in tl or "training" in tl):
                results["Type"] = "Sports Performance Academy"
            elif "test prep" in tl or "sat prep" in tl or "act prep" in tl:
                results["Type"] = "Academic Test Prep & Tutoring"
            elif "elementary" in tl and "high school" in tl:
                results["Type"] = "Independent K-12 School"
            elif "high school" in tl or "upper school" in tl:
                results["Type"] = "Independent High School"
            elif "middle school" in tl:
                results["Type"] = "Independent Middle School"
            elif "elementary" in tl or "lower school" in tl:
                results["Type"] = "Independent Elementary School"
            elif "academy" in tl:
                results["Type"] = "Independent Academy"
            else:
                results["Type"] = "Independent School"

            soup_home = BeautifulSoup(await page.content(), "html.parser")
            for attr in [{"name": "description"}, {"property": "og:description"}]:
                tag = soup_home.find("meta", attr)
                if tag and tag.get("content"):
                    results["Tagline"] = tag["content"][:200]
                    break
            if not results["Tagline"]:
                results["Tagline"] = extract(r"([A-Z][^.]{30,150}\.)")

            results["Quote"] = scrape_quote_from_soup(soup_home)

            # Images — use module-level harvest_images (scrolls + collects all sources)
            image_set = set()
            await harvest_images(page, url, image_set)
            results["Images"] = [img for img in image_set if img][:10]

        except Exception as e:
            print(f"Home page error: {e}")
            await browser.close()
            return None

        # ── SUBPAGE CRAWL 
        soup  = BeautifulSoup(await page.content(), "html.parser")
        links = soup.find_all("a", href=True)
        queue     = []
        seen_urls = {url.rstrip("/")}

        targets = {
            "about": [
                "about", "history", "welcome", "mission", "who-we-are",
                "our-story", "overview", "vision", "values", "ethos",
            ],
            "outcomes": [
                "results", "destination", "university", "leavers", "beyond",
                "sixth-form", "exam", "college", "achievements", "khda",
                "inspection", "academic-results",
            ],
            "fees": [
                "fees", "tuition", "financial", "cost", "charges",
                "fee-structure", "school-fees", "annual-fees", "pricing",
            ],
            "admission": [
                "admissions", "apply", "entry", "register", "enroll",
                "13-entry", "16-entry", "how-to-apply", "application",
                "join-us", "registration", "enrolment", "open-day",
            ],
            "facilities": [
                "facilities", "campus", "co-curriculum", "sport", "arts",
                "library", "infrastructure", "resources", "activities",
                "co-curricular", "extracurricular",
            ],
        }

        for link in links:
            href, text = link["href"], link.get_text().lower().strip()
            full_url   = urljoin(url, href).split("#")[0].rstrip("/")
            if full_url not in seen_urls and url.split("/")[2] in full_url:
                if any(k in text or k in href.lower() for cat in targets.values() for k in cat):
                    queue.append((full_url, text))
                    seen_urls.add(full_url)

        # Global sentence-level dedup — no sentence may appear in more than one field
        used_sentences: set = set()

        def _register(text: str):
            """Mark all sentences in text as used."""
            if not text:
                return
            for s in re.split(r'(?<=[.!?])\s+', text.strip()):
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and len(key) > 20:   # only register meaningful sentences
                    used_sentences.add(key)

        def dedup_text(text: str) -> str:
            """Return only sentences not yet seen, and register new ones."""
            if not text:
                return text
            parts = re.split(r'(?<=[.!?])\s+', text.strip())
            fresh = []
            for s in parts:
                key = re.sub(r'\s+', ' ', s.strip().lower())
                if key and key not in used_sentences and not is_junk(s):
                    fresh.append(s.strip())
                    used_sentences.add(key)
            return " ".join(fresh)

        def assign_field(field_name: str, text: str, max_sent: int = 2):
            """Assign to results[field_name] only if empty, after deduping."""
            if results.get(field_name):
                return
            cleaned = dedup_text(clean_to_sentences(text, max_sent))
            if cleaned:
                results[field_name] = cleaned

        # Register ALL homepage content so subpages can't repeat any of it
        for _field in ["About", "Tagline", "Philosophy", "Outcomes", "Admissions", "Quote"]:
            _register(results.get(_field, ""))
        for _perf_val in results.get("Performance", {}).values():
            _register(_perf_val)

        # used_paras_global — prevents same raw paragraph appearing in multiple metrics
        used_paras_global: set = set()
        # Pre-register any performance values already scraped from homepage
        for _pv in results.get("Performance", {}).values():
            if _pv:
                used_paras_global.add(re.sub(r'\s+', ' ', _pv.strip().lower())[:200])

        for target_url, link_text in queue[:15]:
            try:
                await page.goto(target_url, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(2)
                main          = page.locator("main").first
                content_scope = main if await main.is_visible() else page
                paragraphs    = await content_scope.locator("p").all_text_contents()
                clean_paras   = [
                    t.strip() for t in paragraphs
                    if len(t.strip()) >= 80
                    and not any(x in t.lower() for x in ["cookie", "privacy", "menu", "navigation", "footer"])
                    and not is_junk(t)   # ← remove error messages, UI strings, boilerplate
                ]
                global_text_pool.extend(clean_paras)
                full_text = " ".join(clean_paras)
                page_text = await page.inner_text("body")
                sub_soup  = BeautifulSoup(await page.content(), "html.parser")

                # Harvest images from every subpage
                await harvest_images(page, url, image_set)
                results["Images"] = [img for img in image_set if img][:10]

                if any(k in link_text or k in target_url for k in targets["about"]):
                    # Filter out paragraphs that start with form/navigation content
                    clean_about = [p for p in clean_paras if not is_junk(p)
                                   and not p.lower().startswith(("please fill", "please note",
                                   "the admissions team", "fill the", "click here", "select"))]
                    assign_field("About", " ".join(clean_about), 2)
                    teach_kws  = ["curriculum", "teaching", "learning", "approach",
                                  "education", "ethos", "method", "ib", "igcse", "philosophy"]
                    teach_para = [p for p in clean_paras if any(k in p.lower() for k in teach_kws)]
                    assign_field("Philosophy", " ".join(teach_para), 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)
                    if not results["Founded"]:
                        sub_cands = extract_founded_from_text(page_text)
                        if sub_cands:
                            sub_cands.sort(key=lambda x: (-x[0], x[1]))
                            results["Founded"] = str(sub_cands[0][1])
                    if not results["City"]:
                        results["City"] = detect_city(url, page_text)

                # ── Students + Ratio: search EVERY subpage
                if not results["Students"]:
                    for pat in student_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            num_str = m.group(1).replace(",", "")
                            try:
                                num = int(num_str)
                                # Sanity: school enrollment between 50 and 10,000
                                if 50 <= num <= 10000:
                                    results["Students"] = num_str + ("+" if "+" in m.group(0) else "")
                                    break
                            except ValueError:
                                pass
                if not results["Ratio"]:
                    for pat in ratio_patterns:
                        m = re.search(pat, page_text, re.I)
                        if m:
                            results["Ratio"] = m.group(1).replace(" ", "")
                            break

                if any(k in link_text or k in target_url for k in targets["outcomes"]):
                    assign_field("Outcomes", full_text, 2)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

                if any(k in link_text or k in target_url for k in targets["fees"]):
                    fee_val = extract_fee_validated(page_text)
                    if fee_val and not results["Fees"]:
                        results["Fees"] = fee_val

                if any(k in link_text or k in target_url for k in targets["admission"]):
                    adm_kws  = ["apply", "application", "admission", "register",
                                "assessment", "entry", "enroll", "enrolment", "registration"]
                    filtered = [p for p in clean_paras if any(k in p.lower() for k in adm_kws)]
                    assign_field("Admissions", " ".join(filtered), 2)
                    dl = re.search(
                        r'(?:deadline|closes?|closing date)\s*[:\-]?\s*([^\n.]{10,60})',
                        page_text, re.I
                    )
                    if dl and results["AppDeadline"].startswith("Rolling"):
                        val = dl.group(1).strip()[:80]
                        has_month = re.search(
                            r'\b(january|february|march|april|may|june|july|august|'
                            r'september|october|november|december|jan|feb|mar|apr|'
                            r'jun|jul|aug|sep|oct|nov|dec)\b', val, re.I)
                        has_date = re.search(r'\b\d{1,2}[\/\-]\d{1,2}', val)
                        if (has_month or has_date) and not is_junk(val):
                            results["AppDeadline"] = val
                    if not results["Ages"]:
                        for pat in age_patterns:
                            m = re.search(pat, page_text, re.I)
                            if m:
                                try:
                                    g = m.groups()
                                    if len(g) >= 2 and g[0] and g[1]:
                                        lo, hi = int(g[0]), int(g[1])
                                        if lo < hi and lo >= 2 and hi <= 25:
                                            results["Ages"] = f"{lo}\u2013{hi}"
                                except Exception:
                                    pass
                                if results["Ages"]:
                                    break

                if any(k in link_text or k in target_url for k in targets["facilities"]):
                    perf_kws = {
                        "Coaching Credentials":   ["qualified teacher", "experienced staff",
                                                   "certified", "credentials", "housemaster",
                                                   "instructor", "faculty", "expertise"],
                        "Student Wellbeing":      ["wellbeing", "pastoral", "counselor",
                                                   "mental health", "welfare", "house system",
                                                   "nurturing", "support programme"],
                        "Academic Integration":   ["curriculum", "igcse", "a-level", "ib",
                                                   "programme of study", "academic programme",
                                                   "learning outcomes", "subject"],
                        "Competitive Pathway":    ["university", "destination", "oxford",
                                                   "cambridge", "russell group", "results",
                                                   "examination", "achievement", "ranking"],
                        "Facilities & Resources": ["facilities", "campus", "library",
                                                   "laboratory", "studio", "theatre",
                                                   "swimming pool", "gymnasium", "court"],
                        "Ongoing Accountability": ["progress", "tracking", "monitoring",
                                                   "feedback", "report", "review",
                                                   "assessment cycle", "parent portal"],
                    }
                    for key, kws in perf_kws.items():
                        if not results["Performance"].get(key):
                            matched = [
                                p for p in clean_paras
                                if any(k in p.lower() for k in kws)
                                and p not in used_paras_global   # global dedup
                                and not is_junk(p)
                            ]
                            if matched:
                                chosen = matched[:1]
                                val = dedup_text(clean_to_sentences(" ".join(chosen), 2))
                                if val:
                                    results["Performance"][key] = val
                                    used_paras_global.update(chosen)
                    if not results["Quote"]:
                        results["Quote"] = scrape_quote_from_soup(sub_soup)

            except Exception as e:
                print(f"  Skipping {target_url}: {e}")
                continue

        # ── APPLY FALLBACKS 
        results["Images"] = list(image_set)[:10]  # refresh after subpage harvesting
        for field, fallback in FALLBACKS.items():
            if not results.get(field):
                results[field] = fallback

        if not results["City"]:
            results["City"] = "Miami, FL"

        # ── FALLBACK PERFORMANCE
        for metric in [
            "Coaching Credentials", "Student Wellbeing", "Academic Integration",
            "Competitive Pathway", "Facilities & Resources", "Ongoing Accountability",
        ]:
            if not results["Performance"].get(metric):
                results["Performance"][metric] = generate_fallback(metric, global_text_pool)

        # ── QUOTE: AI fallback
        if not results["Quote"]:
            print(f"  No quote found on site — generating AI quote…")
            results["Quote"] = generate_ai_quote(
                results["Name"], results["Type"], results["City"], results["About"]
            )

        await browser.close()

        # ── FIX #3: OpenAI gap-fill — runs after scraping, fills only missing/N/A fields
        print(f"\n  Running OpenAI gap-fill for missing fields…")
        results = openai_fill_missing_fields(results)

        # ── POST-FILL DEDUP: ensure AI-filled performance fields don't repeat each other
        seen_perf = set()
        for metric in ["Coaching Credentials","Student Wellbeing","Academic Integration",
                       "Competitive Pathway","Facilities & Resources","Ongoing Accountability"]:
            val = results["Performance"].get(metric, "")
            if not val or val == "N/A":
                continue
            key = re.sub(r'\s+', ' ', val.strip().lower())[:120]
            if key in seen_perf:
                results["Performance"][metric] = ""  # will be re-filled by AI next run
            else:
                seen_perf.add(key)

        # ── POST-PROCESSING SANITIZATION
        PERF_METRICS = ["Coaching Credentials","Student Wellbeing","Academic Integration",
                        "Competitive Pathway","Facilities & Resources","Ongoing Accountability"]
        for field in ["Name","Tagline","Founded","City","Ages","Students","Ratio",
                      "Fees","Type","About","Philosophy","Outcomes","Admissions","Quote"]:
            if results.get(field):
                results[field] = re.sub(r'\s+', ' ', str(results[field])).strip()
        for field in ["About","Philosophy","Outcomes","Admissions"]:
            if results.get(field) and len(results[field].split()) > 50:
                results[field] = clean_to_sentences(results[field], 2)
        for metric in PERF_METRICS:
            val = results["Performance"].get(metric,"")
            if val and len(val.split()) > 50:
                results["Performance"][metric] = clean_to_sentences(val, 2)

        # ── PRINT SUMMARY 
        SEP        = "—" * 50
        city_short = results["City"].split(",")[0]
        print(f"\n{SEP}")
        print(f"Elite › {city_short} › {results['Name']}")
        print(SEP)
        print(
            f"\n{results['Ages'] or 'N/A':<14}{results['Students'] or 'N/A':<14}"
            f"{results['Ratio'] or 'N/A':<14}{results['Founded'] or 'N/A'}"
        )
        print(f"{'Ages':<14}{'Students':<14}{'Ratio':<14}Founded\n")
        print(f"Images Found: {len(results['Images'])}")
        for i, img in enumerate(results["Images"], 1):
            print(f"  {i}. {img}")
        print(f"\nAbout {results['Name']}\n{results['About']}")
        print(f"\nInstitution Details")
        print(f"  Type       : {results['Type']}")
        print(f"  Ages       : {results['Ages'] or 'N/A'}")
        print(f"  Students   : {results['Students'] or 'N/A'}")
        print(f"  Ratio      : {results['Ratio'] or 'N/A'}")
        print(f"  Founded    : {results['Founded'] or 'N/A'}")
        print(f"  City       : {results['City']}")
        print(f"  Annual Fee : {results['Fees']}")
        print(f"\nQuotes\n  \"{results['Quote']}\"")
        print(f"\nHow they teach\n{results['Philosophy']}")
        print(f"\nOutcomes: Where students go\n{results['Outcomes']}")
        print(f"\nAdmissions & How to Apply")
        print(f"  Enquiries Open : {results['EnquiryOpen']}")
        print(f"  App Deadline   : {results['AppDeadline']}")
        print(f"  Process        : {results['Admissions']}")
        print(f"\nHow {results['Name']} Performs")
        for k, v in results["Performance"].items():
            print(f"\n  {k}:\n  {v or 'N/A'}")
        print(f"\n  Location : {results['City']}")
        print(f"  Website  : {results['URL']}")
        print(SEP)

        # ── SAVE 
        wp_code     = generate_wp_blocks(results)
        school_slug = re.sub(r"[^a-z0-9]+", "_", results["Name"].lower())[:35].strip("_")
        out_wp      = f"{school_slug}_wp_blocks.txt"
        out_json    = f"{school_slug}_data.json"

        with open(out_wp, "w", encoding="utf-8") as f:
            f.write(wp_code)

        save_data = {k: v for k, v in results.items() if k != "Images"}
        save_data["ImageCount"] = len(results.get("Images", []))
        save_data["ImageURLs"]  = results.get("Images", [])
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)

        print(f"\n WP blocks  → {out_wp}")
        print(f"School JSON → {out_json}")
        return results


# ── URLS

school_urls = [
    "https://ransomeverglades.org",
    "https://gulliverschools.org",
    "https://carrollton.org",
    "https://palmertrinity.org",
    "https://miamicountryday.org",
    "https://www.miamibeachfl.gov/city-hall/parks-and-recreation/parks-facilities-directory/miami-beach-tennis-center/",
    "https://polotraining.org/",
    "https://crandonparktennis.com/",
    "https://princetonreview.com",
    "https://www.imgacademy.com/sport-camps/performance-camp",
    "https://baptisteyoga.com",
    "https://miamimusicproject.org",
    "https://nwsa.mdc.edu/",
]

async def run_multiple_schools():
    all_results = []
    for url in school_urls:
        print(f"\n{'=' * 60}")
        print(f" Scraping: {url}")
        print(f"{'=' * 60}")
        try:
            result = await extract_school_data(url)
            if result:
                all_results.append(result)
            await asyncio.sleep(3)
        except Exception as e:
            print(f" Failed: {url} → {e}")

    master_file = "all_schools_data_miami.json"
    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n{'=' * 60}")
    print(f"All done! {len(all_results)} institutions scraped.")
    print(f"Master JSON → {master_file}")
    print(f"{'=' * 60}")
    return all_results


if __name__ == "__main__":
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as pool:
            future   = pool.submit(asyncio.run, run_multiple_schools())
            all_data = future.result()
    except RuntimeError:
        asyncio.run(run_multiple_schools())


 Scraping: https://ransomeverglades.org
Connecting to: https://ransomeverglades.org...
  ⚠ networkidle failed (TimeoutError), retrying…
  No quote found on site — generating AI quote…
  ✓ AI quote generated for Ransom Everglades School

  Running OpenAI gap-fill for missing fields…
  → OpenAI filling 11 missing field(s): Fees, About, Philosophy, Outcomes, Admissions, Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability
  ✓ OpenAI filled: Fees, About, Philosophy, Outcomes, Admissions, Coaching Credentials, Student Wellbeing, Academic Integration, Competitive Pathway, Facilities & Resources, Ongoing Accountability

——————————————————————————————————————————————————
Elite › Miami › Ransom Everglades School
——————————————————————————————————————————————————

6–12          1246          8:1           1903
Ages          Students      Ratio         Founded

Images Found: 10
  1. https://bbk12e1-cdn.myschoolcdn.com/